# 20.10 Model compression: pruning

Pruning removes parameters whose contribution is small enough that the system prefers speed, size, or simplicity over unused capacity.

Serving constraints provide the reason to remove weights, while regularization and importance scores help decide what can be removed. This notebook compares unstructured and structured magnitude pruning.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2010
rng = np.random.default_rng(SEED)


def prune_by_threshold(weights, threshold):
    weights = np.asarray(weights, dtype=float)
    mask = np.abs(weights) >= threshold
    pruned = weights * mask
    sparsity = 1.0 - np.mean(mask)
    density = np.mean(mask)
    return {"weights": pruned, "mask": mask, "sparsity": float(sparsity), "density": float(density)}


def pruning_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    specs = [
        {"rung": "D1", "name": "four hand weights", "n": 4, "threshold": 0.10, "structured": False},
        {"rung": "D2", "name": "larger clean vector", "n": 20000, "threshold": 0.05, "structured": False},
        {"rung": "D3", "name": "accuracy-sensitive weights", "n": 50000, "threshold": 0.12, "structured": False},
        {"rung": "D4", "name": "small logistic-model-like weights", "n": 30000, "threshold": 0.08, "structured": True},
        {"rung": "D5", "name": "production structured latency simulation", "n": 180000, "threshold": 0.10, "structured": True},
    ]
    workloads = []
    for spec in specs:
        if spec["rung"] == "D1":
            weights = np.array([0.02, -0.50, 0.10, 0.80])
            importance = np.array([0.05, 0.40, 0.30, 0.25])
        else:
            weights = local_rng.normal(0.0, 0.18, size=spec["n"])
            rare_large = local_rng.random(spec["n"]) < 0.03
            weights[rare_large] += local_rng.normal(0.0, 0.90, size=np.sum(rare_large))
            importance = np.abs(weights) + local_rng.gamma(1.5, 0.03, size=spec["n"])
        workloads.append({**spec, "weights": weights, "importance": importance})
    return workloads


def evaluate_pruning(workload, structured=False):
    pruned = prune_by_threshold(workload["weights"], workload["threshold"])
    sparsity = pruned["sparsity"]
    density = pruned["density"]
    removed_importance = np.sum(workload["importance"] * (~pruned["mask"])) / np.sum(workload["importance"])
    base_accuracy = 0.820
    fine_tune_recovery = 0.007 if structured else 0.003
    accuracy_after_prune = base_accuracy - 0.050 * removed_importance + fine_tune_recovery
    accuracy_after_prune = min(base_accuracy, accuracy_after_prune)
    dense_latency = 100.0
    if structured:
        ideal_latency = dense_latency * max(density, 0.25)
        overhead = 6.0
    else:
        ideal_latency = dense_latency
        overhead = 10.0 * sparsity
    latency_samples = rng.normal(ideal_latency + overhead, 3.0, size=5000)
    latency_samples = np.maximum(latency_samples, 1.0)
    memory_units = dense_latency * density
    return {"pruned": pruned, "accuracy_after_prune": float(accuracy_after_prune), "latency_ms": latency_samples, "memory_units": float(memory_units)}


## The concept, built once: magnitude thresholding

Magnitude pruning keeps weights with $|w|\ge \tau$. In the lesson toy vector $0.02,-0.50,0.10,0.80$ with threshold $0.10$, only $0.02$ is pruned, so density is $3/4=0.75$.

In [ ]:

toy_weights = np.array([0.02, -0.50, 0.10, 0.80])
toy = prune_by_threshold(toy_weights, 0.10)
pruned_count = np.sum(~toy["mask"])

assert pruned_count == 1
assert toy["density"] == 0.75
assert toy["weights"].tolist() == [0.0, -0.5, 0.1, 0.8]
print("toy pruned weights", toy["weights"])
print("toy density", toy["density"])


Sparsity and compute are the production proxy: $\text{sparsity}=n_{pruned}/n_{total}$ and $\text{cost}\approx(1-\text{sparsity})C_{dense}$. The lesson uses $7/20=0.35$ sparsity, $0.65\times100=65$ cost, and fine-tune recovery from $0.808$ to $0.815$.

In [ ]:

sparsity = 7 / 20
density = 1 - sparsity
effective_cost = density * 100
base_accuracy = 0.820
accuracy_proxy = base_accuracy - 0.012
recovered_accuracy = accuracy_proxy + 0.007
remaining_gap = base_accuracy - recovered_accuracy

assert sparsity == 0.35
assert effective_cost == 65.0
assert round(accuracy_proxy, 3) == 0.808
assert round(recovered_accuracy, 3) == 0.815
assert round(remaining_gap, 3) == 0.005
print("sparsity", sparsity)
print("effective cost", effective_cost)
print("recovered accuracy", recovered_accuracy)


## The dataset ladder
D1-D5 move from four hand weights to a production-scale structured pruning simulation.

In [ ]:

workloads = pruning_ladder()
for workload in workloads:
    preview = pd.DataFrame({
        "weight": np.round(workload["weights"][:8], 4),
        "importance": np.round(workload["importance"][:8], 4),
    })
    print(workload["rung"], workload["name"], "n=", workload["n"], "threshold=", workload["threshold"])
    print(preview)


In [ ]:

rows = []
outputs = []
for workload in workloads:
    result = evaluate_pruning(workload, structured=workload["structured"])
    outputs.append((workload, result))
    rows.append({
        "rung": workload["rung"],
        "workload": workload["name"],
        "metric_accuracy_after_prune": float(result["accuracy_after_prune"]),
        "sparsity": float(result["pruned"]["sparsity"]),
        "p95_latency_ms": float(np.percentile(result["latency_ms"], 95)),
        "memory_units": float(result["memory_units"]),
    })
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, (workload, result) in enumerate(outputs):
    axes[0, i].hist(result["latency_ms"], bins=35, color="#4c78a8")
    axes[0, i].set_title(workload["rung"])
    axes[0, i].set_xlabel("latency ms")
    axes[0, i].set_ylabel("requests")
    panel_values = [result["accuracy_after_prune"], result["pruned"]["sparsity"], np.percentile(result["latency_ms"], 95) / 100.0]
    axes[1, i].bar(["acc", "sparse", "p95/100"], panel_values, color=["#54a24b", "#e45756", "#f58518"])
    axes[1, i].set_title("operational panel")
summary_x = metrics["sparsity"]
fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary_x, metrics["metric_accuracy_after_prune"], marker="o", label="accuracy after prune")
ax.set_xlabel("sparsity")
ax.set_ylabel("accuracy after prune")
ax.set_title("accuracy after prune vs sparsity")
ax.legend()
plt.show()


## Pitfall on D5: counting zeros as speed
Unstructured zeros can shrink memory but not latency if the runtime still executes dense kernels. Structured or block pruning aligns the sparsity pattern with hardware and makes latency move.

In [ ]:

d5 = workloads[-1]
unstructured_result = evaluate_pruning({**d5, "structured": False}, structured=False)
structured_result = evaluate_pruning(d5, structured=True)
unstructured_p95 = float(np.percentile(unstructured_result["latency_ms"], 95))
structured_p95 = float(np.percentile(structured_result["latency_ms"], 95))
print("unstructured memory units", unstructured_result["memory_units"])
print("unstructured p95 latency ms", unstructured_p95)
print("structured p95 latency ms", structured_p95)
print("latency improvement ms", unstructured_p95 - structured_p95)


## Evaluate it + practice

- Metric: accuracy after prune, with p95 latency and memory guardrails.
- No-skill baseline: keep the dense model unchanged.
- Cheap sanity check: threshold zero should keep all weights and preserve baseline behavior.
- Ablation: use unstructured zeros only and latency does not improve.
- Failure signals: accuracy cliffs, sparse runtime overhead, or channel shapes that hardware cannot exploit.

Practice: Sweep thresholds and plot the accuracy-latency frontier.

Practice: Compare structured and unstructured pruning at the same sparsity.

Practice: Add a fine-tuning recovery parameter and re-evaluate D3.